# Трансформер проти рекурентної мережі: чи варто міняти

> ⏱ **Зошит навчає шістнадцять мереж.** Заміряно: **близько пʼяти хвилин процесорного часу** на
> чотирьох ядрах без відеокарти, в один потік. Стінного часу піде більше — залежно від
> того, чим ще зайнята машина. Це нормально: навчання і є предметом роботи, а остання
> клітинка друкує фактичний час твого прогону.

Питання, на яке ми тут відповідаємо, звучить по-інженерному: **у нас є корпус
українського тексту й задача вгадувати наступне слово. Чи дасть трансформер кращий
результат за рекурентну мережу, і чи окупиться це часом?**

Щоб відповідь була вартою довіри, ми зробимо чотири речі, які легко пропустити:

1. поставимо поруч **дурні базові моделі** — прості таблиці частот, які нічого не
   навчаються; без них будь-яке число нейромережі ні про що не говорить;
2. **доберемо гіперпараметр кожній моделі**, і доберемо на окремій **відкладеній**
   вибірці, щоб жодна модель не підглядала у відповідь;
3. проженемо кожну мережу на **пʼятьох зернах** — різниця, менша за розкид, не є
   різницею, а вузька купа на трьох зернах уже одного разу виявилась оманливою;
4. перевіримо **дві властивості трансформера прямо на числах**: чи справді причинна
   маска ховає майбутнє й чи справді увага сліпа до порядку слів.

Дані справжні: українські переклади інтерфейсів, які лежать у системі.

In [ ]:
import os
# ці чотири рядки мусять стояти ДО імпорту numpy і torch: без фіксації потоків
# time.process_time() рахує ще й очікування потоків один на одного, і час бреше в рази
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

import sys, re, glob, gettext, math, random, time, gc
from collections import Counter
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)          # те саме для torch, уже після імпорту
started_at = time.process_time()  # звідси рахуємо загальний процесорний час зошита

print('python  ', sys.version.split()[0])
print('numpy   ', np.__version__)
print('torch   ', torch.__version__)
print('потоків ', torch.get_num_threads())

## 1 · Звідки беремо текст

У кожній системі з українською локаллю лежать перекладені рядки інтерфейсів —
файли `.mo` у теці `/usr/share/locale/uk/LC_MESSAGES/`. Це справжня українська мова,
написана людьми, а не згенерована формулою. Домен у неї вузький — технічні
повідомлення, короткі речення, багато наказового способу, — і про це варто памʼятати,
коли робитимемо висновки.

Ріжемо текст на слова регулярним виразом, у якому апостроф є **звʼязкою всередині
слова**, а не межею: `зʼєднання` має лишитись одним токеном, а не двома.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"    # після приведення до нижнього регістру
_token_re = re.compile(TOKEN_PATTERN)

def tokenize(text):
    """Ріже рядок на слова. Апостроф — звʼязка всередині слова, а не межа."""
    return _token_re.findall(text.lower())

def load_corpus():
    """Читає всі українські каталоги перекладів, що є на цій машині."""
    documents = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as handle:
                catalog = gettext.GNUTranslations(handle)
            for source, target in catalog._catalog.items():
                # беремо лише змістовні рядки: службові заголовки й короткі підписи
                # для мовної моделі не дають нічого
                if isinstance(source, str) and isinstance(target, str) \
                   and len(target) > 30 and 'Project-Id' not in target:
                    documents.append(target)
        except Exception:
            pass
    return documents

documents = load_corpus()
sentences = [tokenize(d) for d in documents]
lengths = np.array([len(s) for s in sentences])

print('документів   ', len(documents))
print('слововживань ', int(lengths.sum()))
print('словоформ    ', len({w for s in sentences for w in s}))
print('медіана довжини', int(np.median(lengths)), 'слів')

## 2 · Три частини, а не дві

Звична схема «навчальна плюс перевірна» тут не годиться, і причина конкретна.

Кожній моделі ми добиратимемо число: лічильникам — величину згладжування, мережам —
швидкість навчання. Якщо добирати його **на перевірній вибірці**, то перевірна
перестає бути перевіркою: ми показали моделі відповідь і обрали те, що на цій
відповіді виглядає найкраще. Тому вибірок буде **три**:

| частина | навіщо |
|---|---|
| навчальна | на ній моделі вчаться |
| **відкладена** | на ній добираємо гіперпараметри |
| перевірна | її дістаємо один раз наприкінці |

Ще дві дрібниці. Речення коротші за два слова й довші за тридцять відкидаємо: перші
нічого не дають, другі роздувають пачки. І словник рахуємо **тільки за навчальною
частиною** з порогом частоти 5 — усе рідше стає токеном `<unk>`.

In [ ]:
kept = [s for s in sentences if 2 <= len(s) <= 30]

shuffled = list(kept)
random.Random(0).shuffle(shuffled)               # зерно 0: поділ однаковий у всіх
n_train = int(len(shuffled) * 0.9)
train_all, test_words = shuffled[:n_train], shuffled[n_train:]

index = list(range(len(train_all)))
random.Random(11).shuffle(index)
n_holdout = int(len(train_all) * 0.05)
holdout_words = [train_all[i] for i in index[:n_holdout]]
train_words = [train_all[i] for i in index[n_holdout:]]

MIN_COUNT = 5
counts = Counter(w for s in train_words for w in s)
vocab = ['<pad>', '<eos>', '<unk>'] + sorted(w for w, c in counts.items() if c >= MIN_COUNT)
word_to_id = {w: i for i, w in enumerate(vocab)}
PAD, EOS, UNK = 0, 1, 2
V = len(vocab)

def encode(sents):
    return [[word_to_id.get(w, UNK) for w in s] for s in sents]

train_full = encode(train_words)
holdout = encode(holdout_words)
test = encode(test_words)

print('після відсіву', len(kept), 'речень')
print('навчальних   ', len(train_full))
print('відкладених  ', len(holdout))
print('перевірних   ', len(test))
print('словник      ', V, '(разом із <pad>, <eos>, <unk>)')

## 3 · Скорочуємо навчальну частину — і кажемо чому

Одна епоха трансформера на всіх 78 тисячах навчальних реченнях коштує близько двох
хвилин процесорного часу, а нам потрібно шістнадцять навчань: шість на добір швидкості
й ще десять на пʼять зерен двох моделей. Це майже півгодини чистого рахунку, і разом із
рештою в зошит воно не влізе.

Тому ми беремо **8 000 навчальних речень** — фіксовану частину того самого
корпусу. Відкладена й перевірна вибірки лишаються **повними**: різати треба те, на
чому вчаться, а не те, чим міряють. Словник теж лишається тим самим, інакше перплексії
стануть незіставними.

In [ ]:
TRAIN_SIZE = 8000
train = train_full[:TRAIN_SIZE]

print('навчальних беремо  ', len(train), 'із', len(train_full))
print('перевірна лишається', len(test), 'речень')
print('цілей у навчальній ', sum(len(s) + 1 for s in train))

## 4 · База: моделі, які нічого не навчаються

Перш ніж радіти числу нейромережі, треба знати, з чим його порівнювати.

**Перплексія** — метрика, якою міряють мовні моделі. Це середня «розгубленість»:
скільки слів модель фактично тримає за рівноможливі, коли вгадує наступне. Якщо
модель нічого не знає й вибирає навмання зі словника у 8 тисяч слів, її перплексія
дорівнює 8 тисячам. Менше — краще.

**Уніграма** дивиться лише на частоту самого слова й не дивиться на контекст узагалі.
**Біграма** дивиться на одне попереднє слово. Обом потрібне **згладжування**: пари,
якої в навчанні не було, дістали б ймовірність нуль, а нуль у логарифмі дає
нескінченність. Тому до кожного лічильника додають маленьке число `alpha` — і саме
його ми зараз доберемо на відкладеній вибірці.

In [ ]:
def fit_counts(data):
    """Рахує, скільки разів трапилось кожне слово й кожна пара сусідніх слів."""
    unigram = Counter()
    context = Counter()      # скільки разів слово стояло попереду чогось
    bigram = Counter()
    for s in data:
        previous = EOS       # початок речення вважаємо за <eos>
        for word in s + [EOS]:
            unigram[word] += 1
            context[previous] += 1
            bigram[(previous, word)] += 1
            previous = word
    return unigram, context, bigram, sum(unigram.values())

def perplexity_unigram(unigram, total, data, alpha=1.0):
    log_sum, n = 0.0, 0
    for s in data:
        for word in s + [EOS]:
            p = (unigram.get(word, 0) + alpha) / (total + alpha * V)
            log_sum -= math.log(p); n += 1
    return math.exp(log_sum / n)

def perplexity_bigram(context, bigram, data, alpha):
    log_sum, n = 0.0, 0
    for s in data:
        previous = EOS
        for word in s + [EOS]:
            p = (bigram.get((previous, word), 0) + alpha) / (context.get(previous, 0) + alpha * V)
            log_sum -= math.log(p); n += 1; previous = word
    return math.exp(log_sum / n)

unigram, context, bigram, total_words = fit_counts(train)

ALPHAS = [10, 3, 1.0, 0.3, 0.1, 0.03, 0.01, 0.005, 0.003, 0.002, 0.001, 0.0005, 0.0001]
on_holdout = [perplexity_bigram(context, bigram, holdout, a) for a in ALPHAS]
best_alpha = ALPHAS[int(np.argmin(on_holdout))]

print(' добавка |  на відкладеній')
for a, value in zip(ALPHAS, on_holdout):
    mark = '  <-- найкраща' if a == best_alpha else ''
    print(' %-7g | %14.4f%s' % (a, value, mark))
print()
print('увага: мінімум лежить УСЕРЕДИНІ сітки, а не на її краю —')
print('інакше справжнього оптимуму ми б не побачили й сітку треба було б розширювати')

### Суміш двох лічильників

Біграма сильна там, де пару вже бачила, і безпорадна там, де ні. Уніграма навпаки:
вона нічого не знає про контекст, зате знає про кожне слово. Логічно взяти обидві й
змішати:

```
p(слово) = λ · біграма + (1 − λ) · уніграма
```

Вага `λ` — другий гіперпараметр, і його теж добираємо на відкладеній.

In [ ]:
def perplexity_mixture(data, lam, alpha):
    """Суміш біграми з уніграмою. Уніграма зі згладжуванням +1, біграма — з alpha."""
    log_sum, n = 0.0, 0
    for s in data:
        previous = EOS
        for word in s + [EOS]:
            p_bigram = (bigram.get((previous, word), 0) + alpha) / (context.get(previous, 0) + alpha * V)
            p_unigram = (unigram.get(word, 0) + 1.0) / (total_words + V)
            p = lam * p_bigram + (1 - lam) * p_unigram
            log_sum -= math.log(p); n += 1; previous = word
    return math.exp(log_sum / n)

LAMBDAS = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99]
mix_holdout = [perplexity_mixture(holdout, lam, best_alpha) for lam in LAMBDAS]
best_lambda = LAMBDAS[int(np.argmin(mix_holdout))]

print(' лямбда |  на відкладеній')
for lam, value in zip(LAMBDAS, mix_holdout):
    mark = '  <-- найкраща' if lam == best_lambda else ''
    print(' %-6g | %14.4f%s' % (lam, value, mark))

baseline = {
    'уніграма': perplexity_unigram(unigram, total_words, test),
    'біграма, добавка 1.0': perplexity_bigram(context, bigram, test, 1.0),
    'біграма, добавка %g' % best_alpha: perplexity_bigram(context, bigram, test, best_alpha),
    'суміш, лямбда %g' % best_lambda: perplexity_mixture(test, best_lambda, best_alpha),
}
print()
for name, value in baseline.items():
    print('%-26s на перевірній %9.4f' % (name, value))

## 5 · Дві мережі

Тепер самі моделі. Обидві влаштовані однаково зовні: беруть номери слів, перетворюють
їх на вектори по 128 чисел, щось із ними роблять і віддають бали на всі 8 тисяч слів
словника. Різниця — тільки в тому, що саме «щось».

**Рекурентна** несе стан із кроку на крок: щоб порахувати стан після пʼятого слова,
треба знати стан після четвертого.

**Трансформер** такого стану не має. Кожна позиція дивиться на всі позиції ліворуч
одразу — це і є `self-attention`. Щоб модель не підглядала у відповідь, ми даємо їй
**причинну маску**: балам доречності до всіх слів праворуч підставляється мінус
нескінченність, і після softmax вони дістають вагу нуль.

Ще одна деталь: увага не бачить порядку слів (у сумі доданки можна переставляти), тож
до вектора кожного слова додається окремий вектор, що залежить лише від номера позиції.
Без нього трансформер читав би речення як мішок слів.

In [ ]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, width=128, hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, width, padding_idx=PAD)
        self.cell = nn.RNN(width, hidden, num_layers=1, batch_first=True)
        # проєкція потрібна лише тоді, коли стан ширший за вектор слова
        self.project = nn.Linear(hidden, width) if hidden != width else None
        self.output = nn.Linear(width, vocab_size)

    def forward(self, x):
        states, _ = self.cell(self.embedding(x))
        if self.project is not None:
            states = self.project(states)
        return self.output(states)


class TransformerLanguageModel(nn.Module):
    def __init__(self, vocab_size, width=128, heads=4, layers=2, inner=512, max_len=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, width, padding_idx=PAD)
        self.position = nn.Embedding(max_len, width)     # позицію дописуємо окремо
        one_layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads, dim_feedforward=inner, dropout=0.0,
            batch_first=True, norm_first=True, activation='gelu')
        self.body = nn.TransformerEncoder(one_layer, num_layers=layers)
        self.output = nn.Linear(width, vocab_size)

    def forward(self, x):
        length = x.shape[1]
        places = torch.arange(length, device=x.device).unsqueeze(0)
        h = self.embedding(x) + self.position(places)
        # трикутна маска: позиція t бачить себе й усе ліворуч, більше нічого
        mask = nn.Transformer.generate_square_subsequent_mask(length)
        return self.output(self.body(h, mask=mask, is_causal=True))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

torch.manual_seed(0)
example_rnn = RNNLanguageModel(V)
example_transformer = TransformerLanguageModel(V)
# широка рекурентна мережа: стан 520 підібраний так, щоб параметрів вийшло
# приблизно стільки ж, скільки в трансформера — інакше порівняння виміряє розмір
example_wide = RNNLanguageModel(V, hidden=520)

for name, model in (('RNN, стан 128', example_rnn),
                    ('трансформер', example_transformer),
                    ('RNN, стан 520', example_wide)):
    body = count_parameters(model) - model.embedding.weight.numel() \
           - count_parameters(model.output)
    print('%-16s тіло %8d  усього %8d' % (name, body, count_parameters(model)))

## 6 · Як навчаємо й як міряємо

Пачки збираємо з речень **схожої довжини**: інакше коротке речення довелося б
доповнювати нулями до найдовшого в пачці, і половина роботи пішла б у порожнечу.
Самі пачки перемішуємо, щоб порядок навчання не збігався з порядком довжин.

Норму градієнта обрізаємо до одиниці. Без цього окремі зерна час від часу розлітаються,
і розкид по зернах виходить більшим за різницю між моделями — тобто заміряти стає
нічого.

In [ ]:
def make_batches(data, size, seed):
    """Пачки з речень схожої довжини; порядок самих пачок випадковий."""
    order = sorted(range(len(data)), key=lambda j: (len(data[j]), j))
    groups = [order[i:i + size] for i in range(0, len(order), size)]
    random.Random(seed).shuffle(groups)
    return [[data[j] for j in g] for g in groups]

def pack(chunk):
    """Речення пачки в один прямокутник: вхід зсунуто на слово вліво від цілі."""
    width = max(len(s) for s in chunk) + 1
    x = torch.zeros(len(chunk), width, dtype=torch.long)
    y = torch.zeros(len(chunk), width, dtype=torch.long)
    for row, s in enumerate(chunk):
        sequence = [EOS] + s + [EOS]
        x[row, :len(s) + 1] = torch.tensor(sequence[:-1])
        y[row, :len(s) + 1] = torch.tensor(sequence[1:])
    return x, y

def train_one_epoch(model, data, learning_rate, seed, batch=64):
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss(ignore_index=PAD)
    model.train()
    for chunk in make_batches(data, batch, seed):
        x, y = pack(chunk)
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_function(logits.reshape(-1, logits.shape[-1]), y.reshape(-1))
        loss.backward()
        # без обрізання норми окремі зерна розлітаються й розкид з'їдає різницю
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    return model

@torch.no_grad()
def perplexity(model, data, batch=256):
    """Експонента середньої мінус-логарифмічної ймовірності на всіх цілях."""
    model.eval()
    loss_function = nn.CrossEntropyLoss(ignore_index=PAD, reduction='sum')
    total, n = 0.0, 0
    for chunk in make_batches(data, batch, 0):
        x, y = pack(chunk)
        logits = model(x)
        total += loss_function(logits.reshape(-1, logits.shape[-1]), y.reshape(-1)).item()
        n += int((y != PAD).sum())
    return math.exp(total / n)

def build(kind, seed):
    torch.manual_seed(seed)
    if kind == 'rnn':
        return RNNLanguageModel(V)
    return TransformerLanguageModel(V)

print('готово: функції навчання й вимірювання визначені')

## 7 · Добираємо швидкість навчання — обом, і на відкладеній

Швидкість навчання визначає, наскільки сильно мережа посуває ваги після кожної пачки.
Замала — мережа не встигає нічого вивчити за один прохід; завелика — вона перестрибує
потрібне й розхитується.

Це та точка, де порівняння моделей найлегше зіпсувати: якщо покрутити ручку одній
моделі й не покрутити другій, виграє та, якій крутили. Тому сітка **однакова для
обох**, і дивимось ми на **відкладену** вибірку.

In [ ]:
LEARNING_RATES = [0.002, 0.004, 0.008]
grid = {}

for kind in ('rnn', 'transformer'):
    for rate in LEARNING_RATES:
        started = time.process_time()
        model = train_one_epoch(build(kind, 0), train, rate, 0)
        grid[(kind, rate)] = perplexity(model, holdout)
        print('%-12s швидкість %-6g відкладена %9.4f   %5.1f c' %
              (kind, rate, grid[(kind, rate)], time.process_time() - started))
        del model            # кожна модель важить кілька десятків мегабайт
        gc.collect()

best_rate = {}
for kind in ('rnn', 'transformer'):
    best_rate[kind] = min(LEARNING_RATES, key=lambda r: grid[(kind, r)])
    print('%-12s найкраща швидкість %g' % (kind, best_rate[kind]))

## 8 · Пʼять зерен, бо одне число нічого не доводить

Зерно задає початкові ваги мережі й порядок пачок. Дві мережі з різними зернами
навчаються по-різному, і різниця між ними буває більшою, ніж різниця між самими
моделями. Тому кожну модель навчаємо **пʼять разів** і дивимось на медіану та на
**купу** — проміжок від найгіршого зерна до найкращого.

Чому саме пʼять, а не три. Трьох зерен досить, щоб побачити розкид, і замало, щоб
йому вірити: вузька купа на трьох зернах часто виявляється випадковістю, а не
властивістю моделі. Дешева перевірка — додати ще два зерна й подивитись, чи купа
лишилась вузькою.

Правило просте: якщо купи двох моделей **перетинаються**, різниці між моделями немає,
хоч би що казали медіани.

Ось тут ми вперше й востаннє чіпаємо перевірну вибірку.

In [ ]:
SEEDS = (0, 1, 2, 3, 4)
results = {}
for kind in ('rnn', 'transformer'):
    values, seconds = [], []
    for seed in SEEDS:
        started = time.process_time()
        model = train_one_epoch(build(kind, seed), train, best_rate[kind], seed)
        seconds.append(time.process_time() - started)
        values.append(perplexity(model, test))
        print('   %-12s зерно %d  перевірна %9.4f   %5.1f c' %
              (kind, seed, values[-1], seconds[-1]))
        del model            # модель більше не потрібна: звільняємо памʼять одразу
        gc.collect()
    values.sort(); seconds.sort()
    middle = len(values) // 2
    results[kind] = dict(median=values[middle], low=values[0], high=values[-1],
                         seconds=seconds[middle])
    print('%-12s медіана %.4f   купа %.4f…%.4f   час %.1f c\n' %
          (kind, values[middle], values[0], values[-1], seconds[middle]))

## 9 · Складаємо все в одну таблицю

Тепер усі моделі поруч, на одній перевірній вибірці, з однією метрикою.

In [ ]:
rows = list(baseline.items()) + [
    ('RNN, стан 128', results['rnn']['median']),
    ('трансформер', results['transformer']['median']),
]
rows.sort(key=lambda pair: -pair[1])

print('%-28s %12s' % ('модель', 'перплексія'))
for name, value in rows:
    print('%-28s %12.4f' % (name, value))

rnn_median = results['rnn']['median']
transformer_median = results['transformer']['median']
best_counting_name = min(baseline, key=lambda k: baseline[k])
best_counting = baseline[best_counting_name]

print()
print('трансформер проти RNN:          %.4f - %.4f = %.4f'
      % (rnn_median, transformer_median, rnn_median - transformer_median))
print('купи по зернах: RNN %.4f…%.4f   трансформер %.4f…%.4f'
      % (results['rnn']['low'], results['rnn']['high'],
         results['transformer']['low'], results['transformer']['high']))
overlap = not (results['transformer']['high'] < results['rnn']['low']
               or results['rnn']['high'] < results['transformer']['low'])
print('купи перетинаються:', 'ТАК - різниці немає' if overlap else 'ні - різниця справжня')
print()
print('трансформер проти найкращого лічильника (%s):' % best_counting_name)
print('   %.4f - %.4f = %.4f' % (transformer_median, best_counting,
                                 transformer_median - best_counting))
print()
print('час навчання: RNN %.1f c, трансформер %.1f c, тобто в %.2f раза дорожче'
      % (results['rnn']['seconds'], results['transformer']['seconds'],
         results['transformer']['seconds'] / results['rnn']['seconds']))

## 10 · Перевірка перша: чи справді маска ховає майбутнє

Тут легко обманути себе. Причинна маска — це кілька рядків коду, і якщо вона мовчки не
працює, модель просто підглядає у відповідь, дістає прекрасну перплексію на навчанні й
нікуди не годиться на новому тексті. Жодної помилки при цьому не буде.

Перевірити можна прямо: візьмемо речення, змінимо в ньому **слово праворуч** від
позиції `t` і подивимось, чи змінився вихід **на позиції** `t`. Якщо маска працює,
вихід мусить лишитись **побітово тим самим**.

In [ ]:
torch.manual_seed(0)
probe = TransformerLanguageModel(V)
probe.eval()

generator = torch.Generator().manual_seed(5)
sentence = torch.randint(3, V, (1, 10), generator=generator)

with torch.no_grad():
    before = probe(sentence)

changed = sentence.clone()
changed[0, 7] = (changed[0, 7] + 100) % V        # міняємо слово на позиції 8 (індекс 7)
with torch.no_grad():
    after = probe(changed)

# позиції 1-7 (індекси 0-6) стоять ЛІВОРУЧ від зміни — вони не мусили ворухнутись
left_difference = (before[0, :7] - after[0, :7]).abs().max().item()
right_difference = (before[0, 7:] - after[0, 7:]).abs().max().item()

print('найбільша зміна ліворуч від правки: %.10f' % left_difference)
print('найбільша зміна праворуч:           %.4f' % right_difference)
assert left_difference == 0.0, 'маска протікає: майбутнє впливає на минуле!'
assert right_difference > 0.0, 'нічого не змінилось узагалі - правка не спрацювала'
print('✅ маска тримає: слово праворуч не вплинуло на жоден вихід ліворуч')

## 11 · Перевірка друга: чи справді увага сліпа до порядку

Друге твердження теми — увага не бачить порядку слів, бо зважена сума не знає, у
якому порядку складали доданки. Порядок їй повертає окремий вектор позиції.

Перевіримо це буквально: приберемо позиційний вектор, подамо ті самі слова у двох
різних порядках і подивимось, чи стануть виходи просто **переставленими**. Якщо так,
то без позиційного кодування трансформер справді читає речення як мішок слів.

In [ ]:
torch.manual_seed(0)
blind = TransformerLanguageModel(V)
blind.position.weight.data.zero_()      # прибираємо позиційний вектор повністю
blind.eval()

words = torch.randint(3, V, (1, 6), generator=torch.Generator().manual_seed(9))
permutation = [3, 0, 5, 1, 4, 2]
permuted = words[:, permutation]

with torch.no_grad():
    # звертаємось прямо до тіла моделі, оминаючи і позиційний вектор, і маску:
    # маска сама по собі залежить від позиції, тож із нею дослід був би нечистим
    h1 = blind.body(blind.embedding(words))
    h2 = blind.body(blind.embedding(permuted))

# якщо увага сліпа до порядку, то h2 має дорівнювати h1, переставленому так само
expected = h1[:, permutation]
difference = (h2 - expected).abs().max().item()

print('найбільша розбіжність між виходом на переставленому вході')
print('і переставленим виходом на початковому: %.8f' % difference)
assert difference < 1e-4, 'вихід не переставився - десь лишилась залежність від позиції'
print('✅ без позиційного вектора трансформер читає речення як мішок слів')

## 12 · Скільки сигналу доходить до першого слова

І останнє. Головна вада рекурентної мережі — градієнт, що йде назад по ланцюгу й
слабшає на кожному кроці. Поміряємо це в обох моделей однаково: подамо речення з `T`
слів, візьмемо вихід на **останньому** слові, поштовхнемо його назад і подивимось, яка
норма градієнта доходить до вектора **першого** слова.

Мережі беремо **ненавчені**: нас цікавить не якість, а те, з чого навчання починає.
Зерен, як і скрізь, пʼять; беремо медіану.

In [ ]:
def signal_ratio(kind, length, seed):
    """У скільки разів слабший градієнт на першому слові порівняно з останнім."""
    torch.manual_seed(seed)
    if kind == 'rnn':
        model = RNNLanguageModel(V)
    else:
        model = TransformerLanguageModel(V, max_len=length + 1)
    words = torch.randint(3, V, (1, length),
                          generator=torch.Generator().manual_seed(1000 + seed))
    vectors = model.embedding(words)
    vectors.retain_grad()                 # хочемо похідну саме по векторах слів
    if kind == 'rnn':
        states, _ = model.cell(vectors)
        if model.project is not None:
            states = model.project(states)
        logits = model.output(states)
    else:
        places = torch.arange(length).unsqueeze(0)
        h = vectors + model.position(places)
        mask = nn.Transformer.generate_square_subsequent_mask(length)
        logits = model.output(model.body(h, mask=mask, is_causal=True))
    logits[0, -1].sum().backward()        # штовхаємо назад тільки з останнього слова
    norms = vectors.grad[0].norm(dim=1)
    return float(norms[-1]), float(norms[0])

print('%-6s %14s %14s' % ('слів', 'RNN', 'трансформер'))
for length in (10, 25, 50, 100):
    row = []
    for kind in ('rnn', 'transformer'):
        ratios = []
        for seed in SEEDS:
            last, first = signal_ratio(kind, length, seed)
            ratios.append(float('inf') if first == 0 else last / first)
        ratios.sort()
        row.append(ratios[len(ratios) // 2])
    text = ['нуль' if not math.isfinite(v) else ('%.4g' % v) for v in row]
    print('%-6d %14s %14s' % (length, text[0], text[1]))

## 13 · Що ми дізналися

Три висновки, і кожен спирається на надруковане вище.

**Перший.** Порівняння двох архітектур має сенс лише разом із купою по зернах — і саме
її надрукував рядок «купи перетинаються». Якщо вони не перетинаються, різниця між
трансформером і рекурентною мережею справжня; якщо перетинаються, її на цих даних немає,
хоч би що казали медіани. Ціна названа окремо: одна епоха трансформера коштує дорожче,
бо в ньому більше параметрів і більше роботи на позицію.

**Другий, і він незручний.** Найкращий результат дає не нейромережа, а **суміш двох
таблиць частот**, у якій дібрано два числа й немає жодного навчання. Це не привід
сказати «трансформер поганий». Це вимір того, чого бракує: **даних**. Наш корпус —
приблизно як одна товста книжка, і в такому обсязі запамʼятати вигідніше, ніж
узагальнити. Перевага трансформера здобувається там, де запамʼятати неможливо, — на
корпусах у тисячі разів більших.

**Третій.** Дві властивості, про які зазвичай читають словами, ми перевірили числами:
маска справді не пропускає майбутнє в минуле (розбіжність рівно нуль), а увага без
позиційного вектора справді не розрізняє порядку слів.

І ще одне спостереження, яке варто забрати з собою: **найбільше на результат вплинув
не вибір архітектури, а добір гіперпараметра базі**. Біграма зі згладжуванням 1.0 і
біграма з дібраним згладжуванням — це та сама модель, і між ними прірва. Модель, якій
не дали покрутити ручку, завжди програє тій, якій дали.

## Завдання

### 🟢 Рівень 1 — База

Додай до порівняння **триграму** — модель, що дивиться на два попередні слова.
Добери їй згладжування на відкладеній вибірці такою самою сіткою, як біграмі.

**Зроблено, якщо:** у таблиці зʼявився рядок триграми, і ти можеш сказати одним
реченням, чому вона не перемогла біграму, спираючись на частку контекстів-пар, яких у
навчальній частині не було.

### 🟡 Рівень 2 — Плюс

Проведи трансформер по сходинках корпусу: навчи його на 4 000, 8 000 і на всіх
доступних навчальних реченнях (три зерна на кожну точку — або одне, якщо шкода часу,
але тоді так і скажи). Поруч порахуй суміш лічильників на тих самих даних.

**Зроблено, якщо:** є графік із двома кривими «перплексія від розміру навчальної
частини» і висновок про те, яка з них спадає швидше — з числами, а не на око.

### 🔴 Рівень 3 — Виклик

Перевір, чи виграє трансформер тому, що він трансформер, а не тому, що він більший.
Навчи `RNNLanguageModel(V, hidden=520)` — у неї приблизно стільки ж параметрів,
скільки в трансформера, — на тих самих даних, із власним добором швидкості й пʼятьма
зернами.

**Зроблено, якщо:** названо медіану й купу широкої рекурентної мережі, сказано, чи
перетинається її купа з купою трансформера, і зроблено висновок про те, чим саме
пояснюється різниця — розміром чи будовою. Якщо пʼять зерен не влазять у твій час,
візьми три — але тоді так і напиши й не роби висновку про різницю, меншу за купу.

### Підказки

- Триграма ділить дані на набагато дрібніші купки: на кожен різний контекст припадає
  в рази менше прикладів, ніж у біграми. Порахуй це число — воно й буде поясненням.
- Малюючи криву від розміру, став розмір по горизонталі в логарифмічній шкалі:
  сходинки 4 000 / 8 000 / 16 000 інакше зіллються біля лівого краю.
- Широка рекурентна мережа навчається помітно довше за вузьку. Перш ніж запускати
  три зерна, заміряй одне й помнож.

In [ ]:
print('процесорного часу на весь зошит: %.1f c' % (time.process_time() - started_at))